# SOAP on simple molecular examples

This notebook builds a **very small and controlled example** to see how a SOAP descriptor behaves when:

1. the **geometry** of a molecule changes,
2. the molecule is **rotated or translated**, and
3. one atom is changed from **C** to **N**.

The goal is not to build a realistic machine-learning workflow, but to understand the meaning of the descriptor in a simple setting.


## Reference and context

This notebook is closely connected to the discussion around molecular similarity in:

**De, Bartók, Csányi and Ceriotti**,  
*Comparing molecules and solids across structural and alchemical space*  
[https://arxiv.org/abs/1601.04077](https://arxiv.org/abs/1601.04077)

The toy example below is useful to think about the type of structural comparison discussed around **equation (9)** in that work.


## What we want to test

We compare two small sets of dimers:

- `samples0`: all structures are **CC** dimers,
- `samples`: one structure is changed from **CC** to **CN**.

For each structure we:

1. compute atom-wise SOAP features,
2. average them over the atoms of the same molecule,
3. compare the resulting molecular fingerprints through a distance matrix.

This is a simple way to connect **local SOAP environments** to a **molecule-level descriptor**.


In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform

from ase import Atoms

from rascal.representations import SphericalInvariants
from rascal.models import Kernel

## Helper functions

These utility functions do three things:

- assign each atom to the molecule it belongs to,
- average atom-wise SOAP vectors into one vector per molecule,
- compute a pairwise distance matrix between molecular descriptors.

The notebook uses Euclidean distances between the averaged SOAP vectors.


In [ ]:
def get_system_tags(frames):
    """Return an array telling to which structure each atom belongs.

    Example:
    if the first molecule has 2 atoms and the second has 3 atoms,
    the returned labels are [0, 0, 1, 1, 1].
    """
    labels = []
    for i, frame in enumerate(frames):
        labels.extend([i] * len(frame))
    return np.array(labels)


def average_soap_per_structure(atom_soap_features, frames):
    """Average atom-wise SOAP features into one descriptor per structure."""
    df = pd.DataFrame(atom_soap_features)
    df["structure"] = get_system_tags(frames)
    return df.groupby("structure").mean().values


def get_distance_matrix(soap_vectors, normalized=True, labels=None):
    """Compute the pairwise Euclidean distance matrix.

    Parameters
    ----------
    soap_vectors : array-like
        One SOAP vector per structure.
    normalized : bool
        If True, divide all distances by the maximum distance so the
        matrix entries lie between 0 and 1.
    labels : list[str] or None
        Optional row/column labels for readability.
    """
    distance = squareform(pdist(soap_vectors))

    if normalized:
        max_val = np.max(distance)
        if max_val > 0:
            distance = distance / max_val

    distance_df = pd.DataFrame(distance, index=labels, columns=labels)

    # Make large matrices easier to inspect in the notebook
    pd.set_option("display.max_columns", None)
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    return distance_df


def get_kernel_matrix(soap_vectors, sigma=0.5, labels=None):
    """Build a simple Gaussian kernel from the distance matrix.

    This is optional in the notebook, but it is useful to remember that
    similarity kernels are often built from descriptor distances.
    """
    distance = squareform(pdist(soap_vectors))
    kernel_matrix = np.exp(-(distance ** 2) / (2 * sigma ** 2))
    return pd.DataFrame(kernel_matrix, index=labels, columns=labels)

## Build a few toy molecules

All examples are very small dimers.  
Some are simply rotated or translated versions of each other, while others have slightly different bond lengths.

This is helpful because SOAP should behave sensibly:

- it should be **insensitive to pure translations and rotations**,
- it should detect **changes in local geometry**,
- and it should also detect **changes in chemical species**.


In [ ]:
samples0 = [
    Atoms('CC', positions=[[0, 0, 0], [1.00, 0, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [0, 1.10, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [1.21, 0, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [0, 1.33, 0]]),
    Atoms('CC', positions=[[1, 0, 1], [1, 1.01, 1]]),
    Atoms('CC', positions=[[0, 0, 0], [1.50, 0, 0]]),
]

samples = [
    Atoms('CC', positions=[[0, 0, 0], [1.00, 0, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [0, 1.10, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [1.21, 0, 0]]),
    Atoms('CC', positions=[[0, 0, 0], [0, 1.33, 0]]),
    Atoms('CN', positions=[[1, 0, 1], [1, 1.01, 1]]),  # here N replaces one C
    Atoms('CC', positions=[[0, 0, 0], [1.50, 0, 0]]),
]

labels = [f"sample_{i}" for i in range(len(samples))]
labels0 = [f"sample0_{i}" for i in range(len(samples0))]

## Define the SOAP descriptor

`SphericalInvariants` computes a SOAP-like rotationally invariant descriptor.

A few important hyperparameters:

- `interaction_cutoff`: how far the local environment is explored,
- `max_radial` and `max_angular`: basis size / descriptor resolution,
- `gaussian_sigma_constant`: width of the Gaussian used to smooth atomic density,
- `global_species`: which chemical species are expected in the dataset.

In the second dataset we include both carbon (`6`) and nitrogen (`7`), so both must appear in `global_species`.


In [ ]:
hypers = {
    "soap_type": "PowerSpectrum",
    "interaction_cutoff": 5.0,
    "max_radial": 6,
    "max_angular": 6,
    "gaussian_sigma_constant": 0.4,
    "gaussian_sigma_type": "Constant",
    "cutoff_smooth_width": 0.5,
    "radial_basis": "GTO",
    "cutoff_function_type": "ShiftedCosine",
    "cutoff_function_parameters": {"width": 0.5},
    "global_species": [6, 7],  # atomic numbers: C = 6, N = 7
}

soap = SphericalInvariants(**hypers)

## Put all structures in a periodic box

Even though these are isolated molecules, some descriptor implementations expect structures
to have a simulation cell and periodic-boundary-condition metadata.

Here we place each molecule in a large enough box so that its periodic images are far away.


In [ ]:
def prepare_structure(structure, cell=(10, 10, 10)):
    """Return a copy of the structure with a box and periodic metadata."""
    structure = structure.copy()
    structure.cell = cell
    structure.pbc = (1, 1, 1)
    structure.wrap()
    return structure

samples0 = [prepare_structure(s) for s in samples0]
samples = [prepare_structure(s) for s in samples]

## Compute atom-wise SOAP features

After `transform`, each atom in each structure has its own SOAP vector.
So if we have 6 dimers, each with 2 atoms, we will obtain 12 atomic feature vectors.


In [ ]:
soap_rep0 = soap.transform(samples0)
soap_rep = soap.transform(samples)

X0 = soap_rep0.get_features(soap)
X = soap_rep.get_features(soap)

In [ ]:
print("Shape of atom-wise SOAP feature matrix for samples0:", X0.shape)
print("Shape of atom-wise SOAP feature matrix for samples :", X.shape)

## From atom-wise descriptors to molecule-wise descriptors

SOAP is fundamentally a **local descriptor**, i.e. one descriptor per atomic environment.

In this notebook, to compare whole molecules, we take the **average SOAP vector over the atoms**
belonging to the same molecule. This is a simple and common way to obtain a structure-level fingerprint.


In [ ]:
avg_soap_samples0 = average_soap_per_structure(X0, samples0)
avg_soap_samples = average_soap_per_structure(X, samples)

print("Shape after averaging (one row per structure):", avg_soap_samples.shape)

## Distance matrix for the all-carbon dataset

Because all structures in `samples0` are made only of carbon, the distances reflect only
differences in **geometry** (bond length, orientation, translation).

A pure rotation or translation should not matter much for SOAP, while a bond-length change should.


In [ ]:
get_distance_matrix(avg_soap_samples0, labels=labels0)

## Distance matrix when one system becomes CN instead of CC

Now we repeat the same analysis after changing one dimer from **CC** to **CN**.

If the SOAP descriptor is encoding both structure and chemical identity correctly, the row/column
corresponding to the CN system should look noticeably different from the all-carbon case.


In [ ]:
get_distance_matrix(avg_soap_samples, labels=labels)

## Optional: look at a similarity kernel instead of a distance matrix

Distances tell us how far structures are from one another.  
A kernel turns this into a **similarity** measure, where values close to 1 indicate very similar structures.


In [ ]:
get_kernel_matrix(avg_soap_samples, sigma=0.5, labels=labels)

## Interpretation

A few points to observe:

1. **Rotation and translation invariance**  
   Structures that differ only by orientation or absolute position should remain very similar.

2. **Sensitivity to geometry**  
   When the bond length changes, the local atomic environments change, and so do the SOAP vectors.

3. **Sensitivity to chemical identity**  
   Replacing one carbon by nitrogen changes the neighbor density and the species channels used by SOAP.
   This should make the corresponding molecular descriptor more distinct.

4. **Averaging is simple but not unique**  
   In this notebook we average atomic SOAP vectors to get a molecule-level fingerprint.
   This is easy to understand, but other pooling or kernel strategies are also possible in practice.
